# Cell Lifetime Diagnostics

This notebook runs the seven-parameter eSOH diagnostic workflow across representative lifetime-aging cells. For each RPT C/20 charge segment, the voltage and dV/dQ traces are fit with material-specific eSOH parameters. The saved outputs retain historical column names (`x100`, `y100`, `si_scale_a`, `si_shift_b`), corresponding to the manuscript symbols `x_n,100`, `x_p,100`, `s_V`, and `U_off`.

The exported per-cell summaries are used to compare degradation pathways and to track how inferred eSOH parameters evolve with Ah throughput.


In [ ]:
from pathlib import Path
import copy
import importlib
import os
import tempfile
import sys

import pandas as pd


def find_repo_root(start: Path) -> Path:
    """Return the repository root that contains the shared code and data folders."""
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "code" / "diagnostic_algorithm_lifetime_crate").is_dir() and (candidate / "data" / "cell_lifetime_data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the current working directory.")


REPO_ROOT = find_repo_root(Path.cwd())
CODE_DIR = REPO_ROOT / "code"
PROJECT_DIR = CODE_DIR / "diagnostic_algorithm_lifetime_crate"
LIFETIME_DATA_DIR = REPO_ROOT / "data" / "cell_lifetime_data"
OUTPUT_ROOT = PROJECT_DIR / "batch_results" / "A01_cell_lifetime_diagnostics"
CACHE_DIR = PROJECT_DIR / "_cache"

# The measured-data loader reads these environment variables when the config
# module is imported. Keeping them repo-relative makes the notebook portable.
os.environ["VOLTAIQ_PROCESSED_ROOT_TYPE1"] = str(LIFETIME_DATA_DIR / "type1")
os.environ["VOLTAIQ_PROCESSED_ROOT_TYPE2"] = str(LIFETIME_DATA_DIR / "type2")
os.environ["VOLTAGE_ONLY_CACHE"] = str(CACHE_DIR)
os.environ["VOLTAGE_ONLY_OUTPUT"] = str(OUTPUT_ROOT)
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "managing_si_burnout_matplotlib"))

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

import diagnostic_algorithm_lifetime_crate.manual_batch_voltage_runner as runmod
importlib.reload(runmod)
from diagnostic_algorithm_lifetime_crate.config import VoltageFitConfig
from diagnostic_algorithm_lifetime_crate.manual_batch_voltage_runner import run_manual_batch

# Example cells from the 25 degC, 0--100% SoC, 25 psi aging condition.
CELLS = [1, 35, 60,]

SCHEMA = "auto"
VOLTAIQ_ROOT = None
DIRECTION = "charge"
PROTOCOL_KEYWORD = "C/20"
REFRESH_CACHE = False
SAVE_PLOTS = True

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("Lifetime data folder:", LIFETIME_DATA_DIR)
print("Output folder:", OUTPUT_ROOT)
print("Cells to run:", CELLS)


In [ ]:
# Fitting configuration used for the lifetime eSOH diagnostics.
fit_cfg = VoltageFitConfig()

# Enable the two-parameter effective silicon-OCP deformation used in the paper.
fit_cfg.enable_si_drift = True


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed


def _run_one_cell_thread(cell):
    """Run the lifetime diagnostic workflow for one cell in an isolated fit config."""
    fit_cfg_local = copy.deepcopy(fit_cfg)
    fit_cfg_local.de_workers = 1

    results_df, curves, out_dir = run_manual_batch(
        cell=cell,
        project_dir=PROJECT_DIR,
        voltaiq_root=VOLTAIQ_ROOT,
        schema=SCHEMA,
        direction=DIRECTION,
        protocol_keyword=PROTOCOL_KEYWORD,
        fit_cfg=fit_cfg_local,
        refresh_cache=REFRESH_CACHE,
        save_plots=SAVE_PLOTS,
        out_dir=OUTPUT_ROOT / f"cell{cell:03d}",
    )

    results_df = results_df.copy()
    results_df["cell"] = int(cell)
    return cell, results_df, curves, out_dir


all_results = []
all_curves = {}
all_out_dirs = {}
max_workers = min(len(CELLS), 4)

with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = {ex.submit(_run_one_cell_thread, cell): cell for cell in CELLS}

    for fut in as_completed(futures):
        cell = futures[fut]
        try:
            cell_out, results_df, curves, out_dir = fut.result()
            print(f"\n===== Finished cell {cell_out:03d} =====")
            all_results.append(results_df)
            all_curves[int(cell_out)] = curves
            all_out_dirs[int(cell_out)] = out_dir
        except Exception as e:
            print(f"\n===== Failed cell {cell:03d} =====")
            print(repr(e))

results_all_df = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()
results_all_df.head()


In [ ]:
# Compact table of the fitted eSOH quantities and silicon-OCP deformation parameters.
# Internal columns x100/y100/si_scale_a/si_shift_b correspond to
# x_n,100/x_p,100/s_V/U_off in the manuscript notation.
summary_cols = [c for c in [
    "cell", "rpt_seq", "rpt_key", "Ah_throughput", "rmse_v_global", "rmse_dvdq_global",
    "Cn_Si", "Cn_Gr", "Cn", "Cp", "LLI", "x100", "y100", "si_scale_a", "si_shift_b",
] if c in results_all_df.columns]

results_all_df[summary_cols]


In [ ]:
# Display the lifetime OCP-deformation and parameter-trend plots generated above.
from IPython.display import Image, display

for cell, out_dir in sorted(all_out_dirs.items()):
    print(f"\n--- cell {cell:03d} ---")
    for name in [
        f"cell{cell:03d}_si_ocp_lifetime_gradient.png",
        f"cell{cell:03d}_parameter_trends_1x3.png",
    ]:
        p = out_dir / name
        print(p)
        if p.exists():
            display(Image(filename=str(p)))
